In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.data import load_region_df
from src.soc_engine import BatterySpec
from src.forecaster import (
    train_houston_price_model,
    generate_forecast_range,
)
from src.dam_bidder import optimise_dam_bid, run_dam_campaign

passed = 0
failed = 0

def check(condition, msg_pass, msg_fail):
    global passed, failed
    if condition:
        print(f"✅ {msg_pass}")
        passed += 1
    else:
        print(f"❌ {msg_fail}")
        failed += 1

df_coast   = load_region_df("coast")
spec       = BatterySpec(power_mw=100, duration_hours=4)
test_start = pd.Timestamp("2025-01-01")

print("Training Houston price model for mini campaign...")
model_result = train_houston_price_model(df_coast, test_start)

mini_start = pd.Timestamp("2025-06-01")
mini_end   = pd.Timestamp("2025-06-03")
mini_fcst  = generate_forecast_range(model_result, df_coast, mini_start, mini_end)
mini_camp  = run_dam_campaign(mini_fcst, df_coast, spec)

# ── Test C1 — SoC carries across days in campaign ─────────────────────────
if len(mini_camp) >= 2:
    day1_end   = mini_camp.iloc[0]["ending_soc_pct"]
    day2_start = mini_camp.iloc[1]["starting_soc_pct"]
    check(
        abs(day1_end - day2_start) < 0.01,
        f"Test C1 — Day 1 ending SoC ({day1_end:.2f}%) = Day 2 starting SoC ({day2_start:.2f}%) — continuity confirmed",
        f"Test C1 — SoC discontinuity: Day 1 end={day1_end:.2f}%, Day 2 start={day2_start:.2f}%"
    )
else:
    check(False, "", "Test C1 — need at least 2 days in mini campaign")

# ── Test C2 — SoC violations reduced vs old implementation ────────────────
if len(mini_camp) > 0:
    total_violations = mini_camp["soc_violations"].sum()
    days             = len(mini_camp)
    check(
        total_violations / days < 1.0,
        f"Test C2 — avg violations per day = {total_violations/days:.2f} (< 1.0 after continuity fix)",
        f"Test C2 — still {total_violations/days:.2f} violations/day — continuity fix may not be working"
    )
else:
    check(False, "", "Test C2 — no campaign days")

# ── Test C3 — bid optimiser uses actual starting SoC ──────────────────────
near_full_spec = BatterySpec(
    power_mw=100,
    duration_hours=4,
    initial_soc_pct=95.0,
)
near_full_prices = pd.Series(
    [5.0]*4 + [30.0]*16 + [80.0]*4,
    index=range(24)
)
near_full_bid = optimise_dam_bid(
    forecast_prices=near_full_prices,
    spec=near_full_spec,
    operating_date=pd.Timestamp("2025-06-15"),
    starting_soc_mwh=near_full_spec.energy_capacity_mwh * 0.95,
)
check(
    len(near_full_bid.charge_hours) == 0,
    "Test C3 — battery nearly full: charge window correctly skipped",
    f"Test C3 — battery nearly full but charge hours = {near_full_bid.charge_hours} — feasibility check not working"
)

# ── Test C4 — PF and DAM engines are independent ──────────────────────────
if len(mini_camp) > 0:
    first_start = mini_camp.iloc[0]["starting_soc_pct"]
    check(
        abs(first_start - spec.initial_soc_pct) < 0.01,
        f"Test C4 — campaign starts at spec initial SoC ({spec.initial_soc_pct}%): {first_start:.2f}%",
        f"Test C4 — wrong starting SoC: {first_start:.2f}%"
    )
else:
    check(False, "", "Test C4 — no campaign days")

print(f"\n{'─'*50}")
print(f"Results: {passed} passed / {failed} failed / {passed+failed} total")
print(f"{'─'*50}")